In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries imported successfully!")


All libraries imported successfully!


In [17]:
%pip install mysql

Note: you may need to restart the kernel to use updated packages.


In [18]:

import os
from dotenv import load_dotenv
import mysql.connector

# Load environment variables from .env
load_dotenv()

# Connect to MySQL without hardcoding credentials
connection = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST"),
    user=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    database=os.getenv("MYSQL_DATABASE")
)

print("MySQL connected successfully!")


MySQL connected successfully!


In [19]:
query = """
SELECT *
FROM cars;
"""

df = pd.read_sql(query, connection)

df.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_16940\2741079312.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,id,car_name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,1,Maruti 800 AC,2007,60000.0,70000,Petrol,Individual,Manual,First Owner
1,2,Maruti Wagon R LXI Minor,2007,135000.0,50000,Petrol,Individual,Manual,First Owner
2,3,Hyundai Verna 1.6 SX,2012,600000.0,100000,Diesel,Individual,Manual,First Owner
3,4,Datsun RediGO T Option,2017,250000.0,46000,Petrol,Individual,Manual,First Owner
4,5,Honda Amaze VX i-DTEC,2014,450000.0,141000,Diesel,Individual,Manual,Second Owner


In [20]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 3577
Columns: 9


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3577 entries, 0 to 3576
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             3577 non-null   int64  
 1   car_name       3577 non-null   str    
 2   year           3577 non-null   int64  
 3   selling_price  3577 non-null   float64
 4   km_driven      3577 non-null   int64  
 5   fuel           3577 non-null   str    
 6   seller_type    3577 non-null   str    
 7   transmission   3577 non-null   str    
 8   owner          3577 non-null   str    
dtypes: float64(1), int64(3), str(5)
memory usage: 251.6 KB


In [22]:
df.isnull().sum()

id               0
car_name         0
year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
dtype: int64

In [23]:
df.describe()

,id,year,selling_price,km_driven
count,3577.000000,3577.000000,3.577000e+03,3577.000000
mean,2087.193458,2012.962538,4.739125e+05,69250.545709
std,1254.720507,4.251759,5.093018e+05,47579.940016
min,1.000000,1992.000000,2.000000e+04,1.000000
25%,1026.000000,2010.000000,2.000000e+05,36000.000000
50%,2012.000000,2013.000000,3.500000e+05,60000.000000
75%,3193.000000,2016.000000,6.000000e+05,90000.000000
max,4340.000000,2020.000000,8.900000e+06,806599.000000


In [24]:
X=df.drop("selling_price",axis=1)
y=df["selling_price"]

In [25]:
print("Input features:")
print(X.columns)

print("\nTarget:")
print(y.name)

Input features:
Index(['id', 'car_name', 'year', 'km_driven', 'fuel', 'seller_type',
       'transmission', 'owner'],
      dtype='str')

Target:
selling_price


In [26]:
X = X.drop("id", axis=1)

In [27]:
numeric_features = [
    "year",
    "car_age",
    "km_driven"
]

categorical_features = [
    "fuel",
    "seller_type",
    "transmission",
    "owner"
]

In [28]:
X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

In [29]:
X.head()

,car_name,year,km_driven,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti 800 AC,2007,70000,False,False,False,True,True,False,True,False,False,False,False
1,Maruti Wagon R LXI Minor,2007,50000,False,False,False,True,True,False,True,False,False,False,False
2,Hyundai Verna 1.6 SX,2012,100000,True,False,False,False,True,False,True,False,False,False,False
3,Datsun RediGO T Option,2017,46000,False,False,False,True,True,False,True,False,False,False,False
4,Honda Amaze VX i-DTEC,2014,141000,True,False,False,False,True,False,True,False,True,False,False


In [30]:
print(X.shape)

(3577, 14)


In [31]:
X = X.drop("car_name", axis=1)

In [32]:
X = X.astype(float)

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [34]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [35]:
print(tf.__version__)

2.22.0-rc0


In [36]:
model = Sequential([
    Dense(599, activation="relu"),
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

d:\PROJECTS\ANN_Car_Price\.venv\Lib\site-packages\keras\src\layers\core\dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [37]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=['mse']
)

In [38]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [39]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [40]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=1000,
    batch_size=52,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 452654465024.0000 - mse: 452654465024.0000 - val_loss: 519563018240.0000 - val_mse: 519562985472.0000
Epoch 2/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 450958786560.0000 - mse: 450958786560.0000 - val_loss: 513498185728.0000 - val_mse: 513498185728.0000
Epoch 3/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 428003983360.0000 - mse: 428003950592.0000 - val_loss: 454190628864.0000 - val_mse: 454190628864.0000
Epoch 4/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 309943566336.0000 - mse: 309943566336.0000 - val_loss: 253043752960.0000 - val_mse: 253043752960.0000
Epoch 5/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 155702853632.0000 - mse: 155702837248.0000 - val_loss: 171962253312.0000 - val_mse: 171962253312.0000
Epoch 6/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 132067975168.0000 - mse: 132067975168.0000 - val_loss: 160802586624.0000 - val_mse: 160802586624.0000
Epoch 7/1000
44/44 ━━━━━━━━

In [41]:
y_pred = model.predict(X_test)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [42]:
y_pred = y_pred.flatten()

In [43]:
print(y_pred[:10])

[5.0610812e+05 5.2198544e+05 6.7830838e+05 2.2314692e+06 1.5023888e+05
 2.6375006e+05 1.3899544e+05 7.1380119e+05 9.7797512e+05 3.2183906e+05]


In [44]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)

print("MAE:", mae)

MAE: 184438.13839363653


In [45]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)

print("MSE:", mse)

MSE: 163608256424.06677


In [46]:
from sklearn.metrics import r2_score

r2=r2_score(y_test , y_pred)
r2

0.49211016450348355

In [47]:
rmse = np.sqrt(mse)

print("RMSE:", rmse)

RMSE: 404485.17454174603


In [48]:
print("Model Evaluation")
print("---------------------------")
print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R²   :", r2)

Model Evaluation
---------------------------
MAE  : 184438.13839363653
MSE  : 163608256424.06677
RMSE : 404485.17454174603
R²   : 0.49211016450348355


In [49]:
results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

results.head(20)

,Actual Price,Predicted Price
0,270000.0,5.061081e+05
1,525000.0,5.219854e+05
2,990000.0,6.783084e+05
3,3800000.0,2.231469e+06
4,229999.0,1.502389e+05
5,325000.0,2.637501e+05
6,210000.0,1.389954e+05
7,500000.0,7.138012e+05
8,790000.0,9.779751e+05
9,360000.0,3.218391e+05


In [50]:
results["Error"] = (
    results["Actual Price"] - results["Predicted Price"]
)

results.head(20)

,Actual Price,Predicted Price,Error
0,270000.0,5.061081e+05,-2.361081e+05
1,525000.0,5.219854e+05,3.014562e+03
2,990000.0,6.783084e+05,3.116916e+05
3,3800000.0,2.231469e+06,1.568531e+06
4,229999.0,1.502389e+05,7.976012e+04
5,325000.0,2.637501e+05,6.124994e+04
6,210000.0,1.389954e+05,7.100456e+04
7,500000.0,7.138012e+05,-2.138012e+05
8,790000.0,9.779751e+05,-1.879751e+05
9,360000.0,3.218391e+05,3.816094e+04


In [51]:
query = """
SELECT
    id,
    car_name,
    year,
    km_driven,
    fuel,
    seller_type,
    transmission,
    owner,
    selling_price
FROM cars;
"""

df_v2 = pd.read_sql(query, connection)

print(df_v2.shape)
df_v2.head()

(3577, 9)


C:\Users\HP\AppData\Local\Temp\ipykernel_16940\1089843448.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v2 = pd.read_sql(query, connection)


,id,car_name,year,km_driven,fuel,seller_type,transmission,owner,selling_price
0,1,Maruti 800 AC,2007,70000,Petrol,Individual,Manual,First Owner,60000.0
1,2,Maruti Wagon R LXI Minor,2007,50000,Petrol,Individual,Manual,First Owner,135000.0
2,3,Hyundai Verna 1.6 SX,2012,100000,Diesel,Individual,Manual,First Owner,600000.0
3,4,Datsun RediGO T Option,2017,46000,Petrol,Individual,Manual,First Owner,250000.0
4,5,Honda Amaze VX i-DTEC,2014,141000,Diesel,Individual,Manual,Second Owner,450000.0


In [52]:
df_v2["car_brand"] = (
    df_v2["car_name"]
    .str.strip()
    .str.split()
    .str[0]
    .str.title()
)

print(df_v2[["car_name", "car_brand"]].head(15))

                           car_name  car_brand
0                     Maruti 800 AC     Maruti
1          Maruti Wagon R LXI Minor     Maruti
2              Hyundai Verna 1.6 SX    Hyundai
3            Datsun RediGO T Option     Datsun
4             Honda Amaze VX i-DTEC      Honda
5              Maruti Alto LX BSIII     Maruti
6         Hyundai Xcent 1.2 Kappa S    Hyundai
7          Tata Indigo Grand Petrol       Tata
8          Hyundai Creta 1.6 VTVT S    Hyundai
9          Maruti Celerio Green VXI     Maruti
10          Chevrolet Sail 1.2 Base  Chevrolet
11         Tata Indigo Grand Petrol       Tata
12  Toyota Corolla Altis 1.8 VL CVT     Toyota
13             Maruti Ciaz VXi Plus     Maruti
14      Hyundai Venue SX Opt Diesel    Hyundai


In [53]:
df_v2["car_age"]=2026-df["year"]

In [54]:
df_v2["car_age"]=2026-df["year"]
df_v2.head()

,id,car_name,year,km_driven,fuel,seller_type,transmission,owner,selling_price,car_brand,car_age
0,1,Maruti 800 AC,2007,70000,Petrol,Individual,Manual,First Owner,60000.0,Maruti,19
1,2,Maruti Wagon R LXI Minor,2007,50000,Petrol,Individual,Manual,First Owner,135000.0,Maruti,19
2,3,Hyundai Verna 1.6 SX,2012,100000,Diesel,Individual,Manual,First Owner,600000.0,Hyundai,14
3,4,Datsun RediGO T Option,2017,46000,Petrol,Individual,Manual,First Owner,250000.0,Datsun,9
4,5,Honda Amaze VX i-DTEC,2014,141000,Diesel,Individual,Manual,Second Owner,450000.0,Honda,12


In [55]:
numeric_features=[ "year",
                       "km_driven" ,
                       "car_age"
]

categorical_features=["car_name",
                           "fuel",
                           "seller_type",
                           "transmission",
                           "owner",
                           "car_brand"]

X_v2 = df_v2[numeric_features + categorical_features].copy()
y_v2 = df_v2["selling_price"].copy()

X_v2=pd.get_dummies(
    X_v2,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print( X_v2.shape[1])
X_v2.head()


1532


,year,km_driven,car_age,car_name_Ambassador Classic 2000 Dsz,car_name_Ambassador Grand 1800 ISZ MPFI PW CL,car_name_Audi A4 1.8 TFSI,car_name_Audi A4 2.0 TDI,car_name_Audi A4 2.0 TDI 177 Bhp Premium Plus,car_name_Audi A4 3.0 TDI Quattro,car_name_Audi A4 30 TFSI Technology,...,car_brand_Mg,car_brand_Mitsubishi,car_brand_Nissan,car_brand_Opelcorsa,car_brand_Renault,car_brand_Skoda,car_brand_Tata,car_brand_Toyota,car_brand_Volkswagen,car_brand_Volvo
0,2007,70000,19,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2007,50000,19,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2012,100000,14,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2017,46000,9,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2014,141000,12,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [56]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_v2,X_test_v2,y_train_v2,y_test_v2=train_test_split(
    X_v2,
    y_v2,
    test_size=0.20,
    random_state=42
)

scaler_v2 = StandardScaler()
X_train_v2=scaler_v2.fit_transform(X_train_v2)
X_test_v2 = scaler_v2.transform(X_test_v2)

In [57]:
model_v2=Sequential([
    Dense(128 , activation="relu",input_shape=(X_train_v2.shape[1],)),
    Dense(64 , activation="relu"),
    Dense(32 , activation="relu"),
    Dense(1)
])

model_v2.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_v2 = model_v2.fit(
    X_train_v2,
    y_train_v2,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1

)

Epoch 1/100


d:\PROJECTS\ANN_Car_Price\.venv\Lib\site-packages\keras\src\layers\core\dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


72/72 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 452659478528.0000 - mae: 471160.1562 - val_loss: 519601913856.0000 - val_mae: 473829.9688
Epoch 2/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 451068493824.0000 - mae: 469625.5000 - val_loss: 516092821504.0000 - val_mae: 470342.7500
Epoch 3/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 435556548608.0000 - mae: 454732.2500 - val_loss: 494037237760.0000 - val_mae: 447706.3750
Epoch 4/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 377986023424.0000 - mae: 394879.5625 - val_loss: 436984250368.0000 - val_mae: 380312.8125
Epoch 5/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 279239163904.0000 - mae: 286651.3750 - val_loss: 378380910592.0000 - val_mae: 322152.4375
Epoch 6/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 209799610368.0000 - mae: 228657.8125 - val_loss: 359675297792.0000 - val_mae: 319398.4375
Epoch 7/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 183562043392.0000 - mae: 211390.5625 - val_loss: 35003616

In [58]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred_v2 = model_v2.predict(X_test_v2).flatten()

mae_v2 = mean_absolute_error(y_test_v2, y_pred_v2)
mse_v2 = mean_squared_error(y_test_v2, y_pred_v2)
rmse_v2 = np.sqrt(mse_v2)
r2_v2 = r2_score(y_test_v2, y_pred_v2)

print("ANN Version 2")
print("---------------------------")
print("MAE :", mae_v2)
print("MSE :", mse_v2)
print("RMSE:", rmse_v2)
print("R²  :", r2_v2)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
ANN Version 2
---------------------------
MAE : 386263.6490135832
MSE : 486723264790.3468
RMSE: 697655.548813558
R²  : -0.5109371879494784


In [59]:


print("\nNumber of unique brands:")
print(df_v2["car_brand"].nunique())




Number of unique brands:
29


In [60]:
print("\nMost common brands:")
print(df_v2["car_brand"].value_counts().head(20))


Most common brands:
car_brand
Maruti           1072
Hyundai           637
Mahindra          328
Tata              308
Ford              220
Honda             216
Toyota            170
Chevrolet         151
Renault           110
Volkswagen         93
Nissan             52
Skoda              49
Fiat               32
Audi               31
Datsun             29
Bmw                25
Mercedes-Benz      21
Jaguar              5
Mitsubishi          5
Land                5
Name: count, dtype: int64


In [61]:
print(df_v2["selling_price"].describe())

print("\nMissing values:")
print(df_v2.isnull().sum())

print("\nDuplicate rows:")
print(df_v2.duplicated().sum())

count    3.577000e+03
mean     4.739125e+05
std      5.093018e+05
min      2.000000e+04
25%      2.000000e+05
50%      3.500000e+05
75%      6.000000e+05
max      8.900000e+06
Name: selling_price, dtype: float64

Missing values:
id               0
car_name         0
year             0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
selling_price    0
car_brand        0
car_age          0
dtype: int64

Duplicate rows:
0


In [62]:
print("Training price statistics:")
print(y_train_v2.describe())

print("\nTesting price statistics:")
print(y_test_v2.describe())

Training price statistics:
count    2.861000e+03
mean     4.717467e+05
std      4.936159e+05
min      2.000000e+04
25%      2.000000e+05
50%      3.500000e+05
75%      5.900000e+05
max      8.900000e+06
Name: selling_price, dtype: float64

Testing price statistics:
count    7.160000e+02
mean     4.825667e+05
std      5.679647e+05
min      3.500000e+04
25%      2.000000e+05
50%      3.500000e+05
75%      6.000000e+05
max      8.150000e+06
Name: selling_price, dtype: float64


In [63]:
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Select input features
numeric_features = ["car_age", "km_driven"]
categorical_features = [
    "car_brand", "fuel", "seller_type", "transmission", "owner"
]

X_v3 = df_v2[numeric_features + categorical_features].copy()
y_v3 = df_v2["selling_price"].copy()

# Convert categorical columns into numeric columns
X_v3 = pd.get_dummies(
    X_v3,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print("Input shape:", X_v3.shape)
print("Target shape:", y_v3.shape)
print("Missing values:", X_v3.isnull().sum().sum())

Input shape: (3577, 41)
Target shape: (3577,)
Missing values: 0


In [64]:
X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3, y_v3, test_size=0.20, random_state=42
)

# Scale input features
scaler_v3 = StandardScaler()

X_train_v3 = scaler_v3.fit_transform(X_train_v3)
X_test_v3 = scaler_v3.transform(X_test_v3)

# Transform target prices using logarithms
y_train_log_v3 = np.log1p(y_train_v3)
y_test_log_v3 = np.log1p(y_test_v3)

print("Training rows:", len(X_train_v3))
print("Testing rows:", len(X_test_v3))
print("Data preparation complete!")

Training rows: 2861
Testing rows: 716
Data preparation complete!


In [65]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

# Set random seed for reproducibility
tf.keras.utils.set_random_seed(42)

# Build ANN model
model_v3 = Sequential([
    Input(shape=(X_train_v3.shape[1],)),
    Dense(1000, activation="relu"),
    Dense(500, activation="relu"),
    Dense(360, activation="relu"),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1)
])

# Compile the model
model_v3.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# Stop training when validation loss stops improving
early_stop_v3 = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

model_v3.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 1000)           │        42,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 500)            │       500,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 360)            │       180,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │        46,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 779,437 (2.97 MB)

 Trainable params: 779,437 (2.97 MB)

 Non-trainable params: 0 (0.00 B)

In [66]:
history_v3 = model_v3.fit(
    X_train_v3,
    y_train_log_v3,
    validation_split=0.2,
    epochs=1000,
    batch_size=100,
    callbacks=[early_stop_v3],
    verbose=1
)


Epoch 1/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - loss: 55.9852 - mae: 5.3655 - val_loss: 2.6904 - val_mae: 0.9968
Epoch 2/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 2.1707 - mae: 1.0867 - val_loss: 0.8234 - val_mae: 0.7138
Epoch 3/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.5886 - mae: 0.5799 - val_loss: 0.2800 - val_mae: 0.4187
Epoch 4/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.2808 - mae: 0.3993 - val_loss: 0.2194 - val_mae: 0.3606
Epoch 5/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.1972 - mae: 0.3386 - val_loss: 0.1912 - val_mae: 0.3390
Epoch 6/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.1669 - mae: 0.3133 - val_loss: 0.1759 - val_mae: 0.3232
Epoch 7/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.1505 - mae: 0.2981 - val_loss: 0.1662 - val_mae: 0.3123
Epoch 8/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.1413 - mae: 0.2893 - val_loss: 0.1608 - val_mae: 0.3077
Epoch 9/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms

In [67]:
# Predict prices in log-transformed units
y_pred_log_v3 = model_v3.predict(X_test_v3).flatten()

# Convert predictions back to rupees
y_pred_v3 = np.expm1(y_pred_log_v3)

# Prevent negative predicted prices
y_pred_v3 = np.maximum(y_pred_v3, 0)

print("First 10 actual prices:")
print(y_test_v3.iloc[:10].values)

print("\nFirst 10 predicted prices:")
print(np.round(y_pred_v3[:10], 2))

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
First 10 actual prices:
[ 270000.  525000.  990000. 3800000.  229999.  325000.  210000.  500000.
  790000.  360000.]

First 10 predicted prices:
[4.1344709e+05 3.6201138e+05 1.1472792e+06 3.0889002e+06 2.5030798e+05
 2.9940941e+05 1.4957797e+05 4.8239984e+05 7.7788694e+05 3.6680481e+05]


In [68]:
mae_v3 = mean_absolute_error(y_test_v3, y_pred_v3)
mse_v3 = mean_squared_error(y_test_v3, y_pred_v3)
rmse_v3 = np.sqrt(mse_v3)
r2_v3 = r2_score(y_test_v3, y_pred_v3)

print(f"MAE  : ₹{mae_v3:,.2f}")
print(f"MSE  : {mse_v3:,.2f}")
print(f"RMSE : ₹{rmse_v3:,.2f}")
print(f"R²   : {r2_v3:.4f}")

MAE  : ₹150,063.74
MSE  : 126,937,394,839.04
RMSE : ₹356,282.75
R²   : 0.6059


In [69]:
import sys
print(sys.executable)

d:\PROJECTS\ANN_Car_Price\.venv\Scripts\python.exe
